# Notebook 2: Qubit State Vectors

In the previous notebook we saw that classical probability vectors have **non-negative**
entries that sum to 1. Quantum mechanics replaces those entries with **complex amplitudes**
whose squared magnitudes sum to 1.

This notebook covers:

1. Computational basis states $|0\rangle$ and $|1\rangle$.
2. General qubit states $\alpha|0\rangle + \beta|1\rangle$.
3. The **Born rule**: how amplitudes become probabilities.
4. **Demo A**: Two states with *identical* probabilities but *different* amplitudes -- and a gate that tells them apart.
5. Visualizations with the `static_plots` module.

In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
import numpy as np

from quantum_demo.states import (
    basis_state,
    qubit_state,
    amplitudes_to_probabilities,
)
from quantum_demo.gates import H, apply_gate
from quantum_demo.viz.static_plots import (
    plot_probabilities,
    plot_real_amplitudes,
    plot_classical_vs_quantum_panel,
    plot_qubit_state_2d,
)

## 1. Computational basis states

The simplest qubit states are the **computational basis** states:

$$|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \qquad
|1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$

These are the quantum analogues of a deterministic classical distribution:
all "weight" is concentrated on one outcome.

In [ ]:
ket0 = basis_state(0, 2)
ket1 = basis_state(1, 2)

print("|0> =", ket0)
print("|1> =", ket1)

In [ ]:
# Born rule: probabilities = |amplitude|^2
print("P(|0>) =", amplitudes_to_probabilities(ket0))
print("P(|1>) =", amplitudes_to_probabilities(ket1))

## 2. General qubit states

A general single-qubit state is

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$$

where $\alpha, \beta \in \mathbb{C}$ and $|\alpha|^2 + |\beta|^2 = 1$.

The function `qubit_state(alpha, beta)` automatically normalizes for us.

In [ ]:
# A state with unequal amplitudes
psi = qubit_state(1, 2)  # will be normalized to (1/sqrt(5), 2/sqrt(5))
print("State vector:", psi)
print("Probabilities:", amplitudes_to_probabilities(psi))
print("Probability sum:", amplitudes_to_probabilities(psi).sum())

In [ ]:
# Visualize on the unit circle
fig = plot_qubit_state_2d(psi, title=r'Qubit state $\alpha|0\rangle + \beta|1\rangle$')
fig.show()

## 3. The Born rule: amplitudes to probabilities

The **Born rule** is the bridge from the quantum world to experimental outcomes:

$$P(\text{outcome } i) = |\langle i | \psi \rangle|^2 = |\psi_i|^2$$

This means:
- We square the magnitude of each amplitude to get a probability.
- The probabilities are always non-negative and sum to 1 (because the state is normalized).
- **Information about the sign/phase of the amplitude is lost** in a single measurement.

But the phase is not meaningless -- it affects what happens if we apply further gates *before* measuring.

In [ ]:
# Several states and their Born probabilities
states = {
    '|0>': basis_state(0, 2),
    '|1>': basis_state(1, 2),
    '(|0>+|1>)/sqrt(2)': qubit_state(1, 1),
    '(|0>-|1>)/sqrt(2)': qubit_state(1, -1),
    '(|0>+2|1>)/sqrt(5)': qubit_state(1, 2),
}

for name, state in states.items():
    probs = amplitudes_to_probabilities(state)
    print(f"{name:25s}  amplitudes={np.real(state)}  probs={probs}")

## 4. Demo A: Same probabilities, different amplitudes

This is one of the most important conceptual points in quantum mechanics.

Consider two states:

$$|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle), \qquad
|{-}\rangle = \frac{1}{\sqrt{2}}(|0\rangle - |1\rangle)$$

Both have the **same measurement probabilities**: 50% for $|0\rangle$ and 50% for $|1\rangle$.

A single measurement cannot distinguish them. But they are physically different states.

In [ ]:
# Define |+> and |->
plus  = qubit_state(1, 1)   # (1/sqrt(2))(|0> + |1>)
minus = qubit_state(1, -1)  # (1/sqrt(2))(|0> - |1>)

print("|+> =", plus)
print("|-> =", minus)
print()
print("Probabilities of |+>:", amplitudes_to_probabilities(plus))
print("Probabilities of |->:", amplitudes_to_probabilities(minus))
print()
print("Identical probabilities!", 
      np.allclose(amplitudes_to_probabilities(plus), 
                  amplitudes_to_probabilities(minus)))

In [ ]:
# Visualize the amplitudes side by side -- the sign difference is visible here
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
labels = ['|0>', '|1>']

plot_real_amplitudes(plus, labels=labels, title='|+> amplitudes', ax=axes[0])
plot_real_amplitudes(minus, labels=labels, title='|-> amplitudes', ax=axes[1])

fig.suptitle('Same probabilities, different amplitudes', fontsize=12, y=1.04)
fig.tight_layout()
plt.show()

The bar charts make the difference vivid:
- $|+\rangle$ has both amplitudes **positive** (both bars point up).
- $|-\rangle$ has the $|1\rangle$ amplitude **negative** (bar points down).

Yet squaring these amplitudes gives 0.5 in both cases. The sign is invisible to a single
measurement -- but it is very much *there*, and it has consequences.

### How to tell them apart: apply the Hadamard gate

The Hadamard gate $H$ acts as:

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$

It turns out that $H|+\rangle = |0\rangle$ and $H|-\rangle = |1\rangle$. So even
though $|+\rangle$ and $|-\rangle$ look identical under direct measurement, applying
$H$ first makes them **perfectly distinguishable**.

This is because the sign of the amplitude determines whether the two paths through $H$
**constructively** or **destructively** interfere.

In [ ]:
# Apply H to |+> and |->
H_plus  = apply_gate(plus, H)
H_minus = apply_gate(minus, H)

print("H|+> =", np.real(H_plus).round(10))
print("H|-> =", np.real(H_minus).round(10))
print()
print("Probabilities after H|+>:", amplitudes_to_probabilities(H_plus))
print("Probabilities after H|->:", amplitudes_to_probabilities(H_minus))

In [ ]:
# Visualize: after H, the two states are completely distinguishable
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
labels = ['|0>', '|1>']

plot_probabilities(amplitudes_to_probabilities(H_plus), 
                   labels=labels, title='H|+> probabilities', ax=axes[0])
plot_probabilities(amplitudes_to_probabilities(H_minus), 
                   labels=labels, title='H|-> probabilities', ax=axes[1])

fig.suptitle('After Hadamard: |+> and |-> diverge completely', fontsize=12, y=1.04)
fig.tight_layout()
plt.show()

This is the essential lesson:

> **Amplitudes carry more information than probabilities.**
> Two states can have identical measurement statistics yet behave
> completely differently under further quantum operations.
> The sign/phase of amplitudes is invisible in a single measurement
> but determines the outcome of interference.

## 5. Classical vs. quantum: a side-by-side view

Let's use the `plot_classical_vs_quantum_panel` to see the three layers:
classical probabilities, quantum amplitudes, and quantum (Born) probabilities.

In [ ]:
classical_probs = np.array([0.5, 0.5])

fig = plot_classical_vs_quantum_panel(
    classical_probs,
    minus,
    labels=['|0>', '|1>'],
    title='Classical 50/50 vs quantum |->',
)
fig.show()

The left and right panels are identical -- both show a 50/50 split. But the middle panel
reveals the **negative amplitude** on $|1\rangle$ that classical probability cannot represent.
That hidden sign is what quantum computation exploits.

## 6. Key takeaways

| Property | Classical Probability | Quantum Amplitude |
|----------|----------------------|-------------------|
| **Type** | Non-negative real | Complex number |
| **Normalization** | $\sum_i p_i = 1$ | $\sum_i |\alpha_i|^2 = 1$ |
| **Sign / Phase** | None | Crucial for interference |
| **Information content** | Probabilities only | Probabilities + relative phases |
| **Distinguishability** | Identical distributions = identical states | Same probabilities can correspond to different states |

In the next notebook, we will see how **measurement** collapses a quantum state and how
repeated measurements recover the Born probabilities.